# Módulo 8 · Clase 2 — Neuronas Artificiales: Pesos, Bias y Activaciones

**Curso de Inteligencia Artificial — Nivel básico**

---

## ¿Qué vamos a ver hoy?

| # | Tema |
|---|------|
| 1 | Anatomía de una neurona artificial |
| 2 | Pesos y bias: qué representan |
| 3 | Funciones de activación: por qué las necesitamos |
| 4 | ReLU, Sigmoid, Tanh, Softmax en detalle |
| 5 | Cómo fluyen los datos por una red (forward pass) |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
print("Librerías listas")

---
## 1. Anatomía de una neurona artificial

```
Cada neurona hace exactamente 2 cosas:

  PASO 1 — Combinación lineal:
    z = w₁·x₁ + w₂·x₂ + ... + wₙ·xₙ + b
    z = W·x + b
    (producto punto de pesos con entradas + sesgo)

  PASO 2 — Función de activación:
    a = f(z)
    (introduce NO LINEALIDAD en la red)

La salida 'a' de esta neurona se convierte en la entrada de la siguiente capa.

Sin función de activación:
  z = W·x + b → una red de 100 capas equivale a UNA sola capa lineal.
  f(g(x)) = composición de lineales = lineal → no gana complejidad.

Con función de activación no lineal:
  La red puede aprender funciones arbitrariamente complejas.
  (Teorema de aproximación universal: una red con suficientes neuronas
  puede aproximar CUALQUIER función continua)
```

In [ ]:
# Visualizar una neurona
fig, ax = plt.subplots(figsize=(10, 5))
ax.set_xlim(0, 10); ax.set_ylim(0, 6); ax.axis('off')
ax.set_title('Anatomía de una neurona artificial', fontsize=12, fontweight='bold')

# Entradas
entradas = [(1, 5, 'x₁=2'), (1, 3.5, 'x₂=-1'), (1, 2, 'x₃=0.5')]
pesos_texto = ['w₁=0.3', 'w₂=-0.5', 'w₃=0.8']

for (x, y, txt), w_txt in zip(entradas, pesos_texto):
    circ = plt.Circle((x, y), 0.35, color='lightgreen', ec='gray', lw=1.5, zorder=5)
    ax.add_patch(circ)
    ax.text(x, y, txt, ha='center', va='center', fontsize=9)
    ax.annotate('', xy=(4.35, 3.5), xytext=(x+0.35, y),
                arrowprops=dict(arrowstyle='->', color='gray', lw=1.2))
    mx, my = (x+0.35 + 4.35)/2 - 0.2, (y + 3.5)/2
    ax.text(mx, my, w_txt, ha='center', va='center', fontsize=8,
            color='darkorange', fontweight='bold')

# Sesgo
ax.text(2.5, 1, 'b = 0.1', ha='center', va='center', fontsize=10,
        bbox=dict(boxstyle='round', facecolor='lightyellow', ec='orange'), color='darkorange')
ax.annotate('', xy=(4.35, 3.2), xytext=(3.0, 1.2),
            arrowprops=dict(arrowstyle='->', color='orange', lw=1.2, ls='dashed'))

# Suma
suma = plt.Circle((4.7, 3.5), 0.55, color='lightblue', ec='steelblue', lw=2, zorder=5)
ax.add_patch(suma)
z_val = 0.3*2 + (-0.5)*(-1) + 0.8*0.5 + 0.1
ax.text(4.7, 3.7, 'Σ', ha='center', va='center', fontsize=14)
ax.text(4.7, 3.2, f'z={z_val:.1f}', ha='center', va='center', fontsize=8)

ax.annotate('', xy=(6.45, 3.5), xytext=(5.25, 3.5),
            arrowprops=dict(arrowstyle='->', color='steelblue', lw=2))
ax.text(5.85, 3.75, 'z', ha='center', fontsize=11, color='steelblue')

# Activación
act = plt.Circle((7.0, 3.5), 0.55, color='lightsalmon', ec='tomato', lw=2, zorder=5)
ax.add_patch(act)
a_val = max(0, z_val)  # ReLU
ax.text(7.0, 3.7, 'f(z)', ha='center', va='center', fontsize=10)
ax.text(7.0, 3.2, f'a={a_val:.1f}', ha='center', va='center', fontsize=8)
ax.text(7.0, 2.5, '(ReLU)', ha='center', fontsize=8, color='gray')

ax.annotate('', xy=(9.0, 3.5), xytext=(7.55, 3.5),
            arrowprops=dict(arrowstyle='->', color='tomato', lw=2))
ax.text(8.75, 3.75, 'a (salida)', ha='center', fontsize=10, color='tomato')

# Fórmula
ax.text(5.0, 0.7, 'z = w₁·x₁ + w₂·x₂ + w₃·x₃ + b    →    a = ReLU(z) = max(0, z)',
        ha='center', fontsize=10, style='italic',
        bbox=dict(boxstyle='round', facecolor='white', ec='gray'))

plt.tight_layout(); plt.show()

---
## 2. Pesos y bias: qué representan

```
PESOS (weights) wᵢ:
  Determinan cuánto importa cada entrada.
  w grande y positivo → esa entrada activa la neurona
  w grande y negativo → esa entrada inhibe la neurona
  w ≈ 0 → esa entrada es irrelevante para la neurona

  En una red de 784→128 neuronas: 784×128 = 100 352 pesos
  En GPT-3: 175 000 000 000 parámetros (175 mil millones)

BIAS (sesgo) b:
  Permite desplazar la activación independientemente de la entrada.
  Sin bias: la frontera de decisión pasa SIEMPRE por el origen.
  Con bias: la frontera puede estar en cualquier lugar.
  Análogo al término independiente en y = mx + b.

Inicialización:
  Si todos los pesos son iguales → todas las neuronas aprenden lo mismo (simetría).
  Solución: inicializar pesos con valores aleatorios pequeños.
  Métodos: Glorot/Xavier (para Tanh), He (para ReLU).
```

In [ ]:
# Efecto del peso y el bias en una neurona simple
x = np.linspace(-5, 5, 300)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# 1. Efecto del peso
for w in [0.5, 1.0, 2.0, -1.0]:
    z = w * x
    a = np.maximum(0, z)  # ReLU
    axes[0].plot(x, a, label=f'w={w}, b=0', lw=2)
axes[0].set_title('Efecto del peso (b=0 fijo)')
axes[0].set_xlabel('x'); axes[0].set_ylabel('ReLU(w·x)')
axes[0].legend(fontsize=8); axes[0].grid(True, alpha=0.3)

# 2. Efecto del bias
for b in [-2, 0, 1, 3]:
    z = x + b  # w=1 fijo
    a = np.maximum(0, z)
    axes[1].plot(x, a, label=f'w=1, b={b}', lw=2)
axes[1].set_title('Efecto del bias (w=1 fijo)')
axes[1].set_xlabel('x'); axes[1].set_ylabel('ReLU(x + b)')
axes[1].legend(fontsize=8); axes[1].grid(True, alpha=0.3)

# 3. Inicialización de pesos
n_pesos = 10000
glorot = np.random.uniform(-np.sqrt(6/512), np.sqrt(6/512), n_pesos)  # Glorot
he     = np.random.randn(n_pesos) * np.sqrt(2/256)  # He init
normal = np.random.randn(n_pesos) * 0.01            # N(0, 0.01)

for datos, etiqueta, color in [
    (glorot, 'Glorot/Xavier', 'steelblue'),
    (he, 'He init', 'seagreen'),
    (normal, 'N(0, 0.01)', 'coral'),
]:
    axes[2].hist(datos, bins=50, alpha=0.6, color=color, label=etiqueta, density=True)
axes[2].set_title('Distribuciones de inicialización de pesos')
axes[2].set_xlabel('Valor del peso'); axes[2].set_ylabel('Densidad')
axes[2].legend(fontsize=8); axes[2].grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

---
## 3. Funciones de activación: por qué las necesitamos

```
Sin activación no lineal:
  Capa 1: a₁ = W₁·x + b₁
  Capa 2: a₂ = W₂·a₁ + b₂ = W₂·(W₁·x + b₁) + b₂ = (W₂W₁)·x + c
  → ¡sigue siendo lineal! La red profunda = una sola capa lineal.

Con activación no lineal f:
  Capa 1: a₁ = f(W₁·x + b₁)
  Capa 2: a₂ = f(W₂·a₁ + b₂)
  → composición de funciones no lineales = capacidad expresiva real.

Propiedades deseables:
  1. No lineal (para que la red sea expresiva)
  2. Diferenciable (para poder hacer backpropagation)
  3. No satura fácilmente (evitar gradiente evanescente)
  4. Computacionalmente barata
```

---
## 4. ReLU, Sigmoid, Tanh, Softmax

In [ ]:
z = np.linspace(-6, 6, 400)

# Implementar funciones de activación
def relu(z):    return np.maximum(0, z)
def leaky_relu(z, alpha=0.1): return np.where(z >= 0, z, alpha * z)
def sigmoid(z): return 1 / (1 + np.exp(-z))
def tanh(z):    return np.tanh(z)

# Derivadas
def d_relu(z):    return (z > 0).astype(float)
def d_sigmoid(z): s = sigmoid(z); return s * (1 - s)
def d_tanh(z):    return 1 - np.tanh(z)**2

fig, axes = plt.subplots(2, 4, figsize=(16, 8))

configs = [
    ('ReLU', relu, d_relu, 'steelblue',
     'ReLU(z) = max(0,z)\n\nPro: simple, no satura z>0\nPro: gradiente constante\nCon: "neurona muerta" z<0'),
    ('Leaky ReLU', leaky_relu, None, 'teal',
     'LReLU(z) = z si z≥0, αz si z<0\n\nPro: no muere nunca\nα=0.01 típico'),
    ('Sigmoid', sigmoid, d_sigmoid, 'tomato',
     'σ(z) = 1/(1+e^-z) ∈ (0,1)\n\nPro: salida como probabilidad\nCon: satura (gradiente ≈0)\nUso: clasificación binaria (última capa)'),
    ('Tanh', tanh, d_tanh, 'seagreen',
     'tanh(z) ∈ (-1,1)\n\nPro: centrada en 0\nCon: satura igual que sigmoid\nUso: RNN'),
]

for col, (nombre, f, df, color, notas) in enumerate(configs):
    # Función
    axes[0, col].plot(z, f(z), color=color, lw=2.5)
    axes[0, col].axhline(0, color='black', lw=0.5, ls='--')
    axes[0, col].axvline(0, color='black', lw=0.5, ls='--')
    axes[0, col].set_title(nombre, fontsize=11, fontweight='bold', color=color)
    axes[0, col].set_xlabel('z'); axes[0, col].grid(True, alpha=0.3)

    # Notas
    axes[1, col].text(0.05, 0.95, notas, transform=axes[1, col].transAxes,
                      fontsize=9, va='top', fontfamily='monospace',
                      bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))
    # Derivada
    if df is not None:
        axes[1, col].plot(z, df(z), color=color, lw=2, ls='--', alpha=0.8, label="f'(z)")
    elif nombre == 'Leaky ReLU':
        axes[1, col].plot(z, (z >= 0).astype(float) + 0.01*(z < 0), color=color, lw=2, ls='--', label="f'(z)")
    axes[1, col].axhline(0, color='black', lw=0.5)
    axes[1, col].set_xlabel('z'); axes[1, col].legend(fontsize=8)
    axes[1, col].grid(True, alpha=0.3)

axes[0, 0].set_ylabel('f(z)')
axes[1, 0].set_ylabel('f\'(z) y notas')
plt.suptitle('Funciones de activación y sus derivadas', fontsize=12)
plt.tight_layout(); plt.show()

In [ ]:
# Softmax: activación para multiclase
def softmax(z):
    # Truco numérico: restar el máximo evita overflow
    e = np.exp(z - np.max(z))
    return e / e.sum()

print("Softmax — convierte logits a probabilidades que suman 1")
print("Uso: ÚNICA capa de salida en clasificación multiclase\n")

ejemplos = [
    np.array([2.0, 1.0, 0.1]),           # clase 0 clara
    np.array([0.5, 0.5, 0.5]),           # incertidumbre total
    np.array([10.0, 2.0, 0.5]),          # clase 0 muy segura
    np.array([1.0, 2.0, 3.0]),           # clase 2
]
clases = ['gato', 'perro', 'pájaro']

fig, axes = plt.subplots(1, 4, figsize=(14, 4))
for ax, logits in zip(axes, ejemplos):
    proba = softmax(logits)
    bars  = ax.bar(clases, proba, color=['steelblue','coral','seagreen'])
    ax.set_ylim(0, 1.05)
    ax.set_title(f'logits={np.round(logits,1)}', fontsize=9)
    ax.set_ylabel('Probabilidad')
    for bar, p in zip(bars, proba):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.02,
                f'{p:.2f}', ha='center', fontsize=9)
    ax.text(0.5, 0.92, f'Suma={proba.sum():.2f}', transform=ax.transAxes,
            ha='center', fontsize=8, color='gray')

plt.suptitle('Softmax: logits → probabilidades (siempre suman 1)', fontsize=10)
plt.tight_layout(); plt.show()

---
## 5. Forward pass: cómo fluyen los datos

```
Forward pass = propagación hacia adelante
Los datos fluyen desde la entrada hasta la salida.

Para una red con 2 capas ocultas:

  Entrada: x  (vector de n_features)

  Capa 1:  z₁ = W₁·x + b₁
           a₁ = ReLU(z₁)

  Capa 2:  z₂ = W₂·a₁ + b₂
           a₂ = ReLU(z₂)

  Salida:  z_out = W_out·a₂ + b_out
           ŷ = Softmax(z_out)  (multiclase) o Sigmoid(z_out) (binario)

  Pérdida: L = -Σ yᵢ·log(ŷᵢ)  (cross-entropy)

El objetivo del entrenamiento:
  Ajustar TODOS los pesos y bias para minimizar L.
  Cómo → Backpropagation (Clase 3).
```

In [ ]:
# Forward pass desde cero — red 2→3→2
np.random.seed(42)

class RedNeuronal:
    """MLP simple implementado desde cero (solo forward pass por ahora)"""

    def __init__(self, capas):
        """capas: lista de enteros, ej. [2, 4, 3] = 2 entradas, 4 ocultas, 3 salidas"""
        self.pesos = []
        self.biases = []
        for i in range(len(capas)-1):
            # Inicialización He (para ReLU)
            W = np.random.randn(capas[i+1], capas[i]) * np.sqrt(2.0 / capas[i])
            b = np.zeros(capas[i+1])
            self.pesos.append(W)
            self.biases.append(b)

    def forward(self, x, verbose=False):
        a = x
        for i, (W, b) in enumerate(zip(self.pesos, self.biases)):
            z = W @ a + b
            es_ultima = (i == len(self.pesos) - 1)
            if es_ultima:
                e = np.exp(z - z.max())
                a = e / e.sum()  # Softmax
                nombre_act = 'Softmax'
            else:
                a = np.maximum(0, z)  # ReLU
                nombre_act = 'ReLU'
            if verbose:
                print(f"  Capa {i+1}: z={z.round(3)}, activación={nombre_act}, a={a.round(3)}")
        return a


# Demo: red 2→4→3 con un ejemplo
red = RedNeuronal([2, 4, 3])
x_ejemplo = np.array([1.5, -0.5])

print("Forward pass en red 2→4→3:")
print(f"Entrada: {x_ejemplo}")
salida = red.forward(x_ejemplo, verbose=True)
print(f"\nSalida (probabilidades): {salida.round(4)}")
print(f"Clase predicha: {np.argmax(salida)}")
print(f"Suma de probabilidades: {salida.sum():.6f}  ← siempre 1")

In [ ]:
# Verificar las formas matriciales en una red más grande
print("Formas matriciales en el forward pass:")
print(f"{'Capa':15} {'Entrada':15} {'W shape':15} {'b shape':10} {'Salida'}")
print("-" * 70)

capas = [784, 128, 64, 10]  # red para MNIST
nombres = ['FC1 (ReLU)', 'FC2 (ReLU)', 'Salida (Softmax)']
for i in range(len(capas)-1):
    ent = capas[i]; sal = capas[i+1]
    n_params = ent * sal + sal
    print(f"{nombres[i]:15} ({ent},):          ({sal},{ent}):     ({sal},):    ({sal},)  → {n_params:,} params")

total = sum(capas[i]*capas[i+1] + capas[i+1] for i in range(len(capas)-1))
print(f"\nTotal parámetros: {total:,}")
print(f"Memoria (float32): {total*4/1024:.1f} KB")

---
## Práctica

### Ejercicio 1 — Implementar funciones de activación

**Enunciado:** Implementa las siguientes funciones y sus derivadas:

1. `elu(z, alpha=1.0)` → ELU: `z si z>0`, `alpha*(exp(z)-1) si z≤0`
2. `swish(z)` → Swish: `z * sigmoid(z)`
3. `gelu(z)` → GELU (aproximación): `0.5 * z * (1 + tanh(sqrt(2/π) * (z + 0.044715 * z³)))`

Plotéalas junto a ReLU y comenta sus diferencias.

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 1</summary>

```python
import numpy as np
import matplotlib.pyplot as plt

z = np.linspace(-4, 4, 400)

def relu(z):   return np.maximum(0, z)
def sigmoid(z): return 1 / (1 + np.exp(-z))
def elu(z, alpha=1.0):  return np.where(z > 0, z, alpha * (np.exp(z) - 1))
def swish(z):  return z * sigmoid(z)
def gelu(z):   return 0.5 * z * (1 + np.tanh(np.sqrt(2/np.pi) * (z + 0.044715 * z**3)))

fig, ax = plt.subplots(figsize=(9, 4))
for f, nombre, color in [
    (relu, 'ReLU', 'steelblue'),
    (elu,  'ELU', 'tomato'),
    (swish,'Swish', 'seagreen'),
    (gelu, 'GELU', 'mediumpurple'),
]:
    ax.plot(z, f(z), label=nombre, lw=2)

ax.axhline(0, color='black', lw=0.5, ls='--')
ax.axvline(0, color='black', lw=0.5, ls='--')
ax.set_xlabel('z'); ax.set_ylabel('f(z)')
ax.set_title('Comparativa de funciones de activación modernas')
ax.legend(); ax.grid(True, alpha=0.3)
ax.set_ylim(-1.5, 4)
plt.tight_layout(); plt.show()

print("ReLU:  simple, rápida, puede 'matar' neuronas (z<0 → gradiente=0)")
print("ELU:   suave en z<0, permite gradientes no cero → no muere")
print("Swish: suave, no monótona, funciona mejor en redes muy profundas (Google 2017)")
print("GELU:  usada en BERT y GPT. Suave y estocástica en su versión original")
```
</details>

---
### Ejercicio 2 — Problema del gradiente evanescente

**Enunciado:** Demuestra por qué Sigmoid causa el problema del gradiente evanescente en redes profundas:

1. La derivada de Sigmoid es: `σ'(z) = σ(z)·(1-σ(z))`
2. El valor máximo de σ' es 0.25 (en z=0)
3. Simula backprop: multiplica 20 derivadas de sigmoid con entrada z=0
4. Haz lo mismo con ReLU (derivada = 1 si z>0)
5. ¿Por qué ReLU resuelve el problema?

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 2</summary>

```python
import numpy as np
import matplotlib.pyplot as plt

def sigmoid(z): return 1 / (1 + np.exp(-z))
def d_sigmoid(z): s = sigmoid(z); return s * (1 - s)
def d_relu(z): return float(z > 0)

n_capas = 30
z_val   = 0.5  # entrada a cada capa

grad_sigmoid = [1.0]
grad_relu    = [1.0]

for _ in range(n_capas):
    grad_sigmoid.append(grad_sigmoid[-1] * d_sigmoid(z_val))
    grad_relu.append(grad_relu[-1] * d_relu(z_val))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].semilogy(range(n_capas+1), grad_sigmoid, 'r-o', ms=4, label='Sigmoid')
axes[0].semilogy(range(n_capas+1), grad_relu,    'b-o', ms=4, label='ReLU')
axes[0].set_xlabel('Número de capas'); axes[0].set_ylabel('Magnitud del gradiente (log)')
axes[0].set_title('Gradiente evanescente: Sigmoid vs ReLU')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].text(0.05, 0.95, 
    f"Gradiente Sigmoid tras {n_capas} capas:\n  {grad_sigmoid[-1]:.2e}\n\n"
    f"Gradiente ReLU tras {n_capas} capas:\n  {grad_relu[-1]:.2f}\n\n"
    f"d_sigmoid(0.5) = {d_sigmoid(z_val):.4f} < 0.25\n"
    f"d_relu(0.5)    = {d_relu(z_val):.4f}\n\n"
    f"Tras {n_capas} capas:\n"
    f"  Sigmoid: 0.25^{n_capas} = {0.25**n_capas:.2e}\n"
    f"  → los pesos de las primeras capas\n"
    f"    no aprenden nada (gradiente ≈ 0)\n\n"
    f"  ReLU: 1^{n_capas} = 1.0\n"
    f"  → el gradiente fluye sin atenuarse",
    transform=axes[1].transAxes, fontsize=10, va='top',
    fontfamily='monospace',
    bbox=dict(boxstyle='round', facecolor='lightyellow'))
axes[1].axis('off')
plt.tight_layout(); plt.show()
```
</details>

---
### Ejercicio 3 — Forward pass manual

**Enunciado:** Dado este MLP con pesos conocidos:

```
Arquitectura: 3 → 2 → 1 (clasificación binaria)

W1 = [[0.5, -0.2, 0.1],    b1 = [0.0, 0.5]
      [0.3,  0.8, -0.5]]

W2 = [[0.7, -0.3]]         b2 = [0.1]

Entrada: x = [1.0, 2.0, -1.0]
Capas ocultas: ReLU. Salida: Sigmoid.
```

1. Calcula a mano cada paso (z₁, a₁, z₂, a₂)
2. Verifica con código
3. ¿Cuál es la probabilidad predicha de clase 1?
4. ¿Qué clase predice con umbral 0.5?

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 3</summary>

```python
import numpy as np

# Pesos
W1 = np.array([[0.5, -0.2, 0.1],
               [0.3,  0.8, -0.5]])
b1 = np.array([0.0, 0.5])
W2 = np.array([[0.7, -0.3]])
b2 = np.array([0.1])

x = np.array([1.0, 2.0, -1.0])

# Forward pass paso a paso
z1 = W1 @ x + b1
print(f"z₁ = W1·x + b1 = {z1.round(4)}")

a1 = np.maximum(0, z1)  # ReLU
print(f"a₁ = ReLU(z₁) = {a1.round(4)}")

z2 = W2 @ a1 + b2
print(f"z₂ = W2·a₁ + b2 = {z2.round(4)}")

a2 = 1 / (1 + np.exp(-z2))  # Sigmoid
print(f"a₂ = σ(z₂) = {a2.round(4)}")

clase = int(a2[0] >= 0.5)
print(f"\nProbabilidad clase 1: {a2[0]:.4f}")
print(f"Clase predicha (umbral 0.5): {clase}")

# Verificación manual:
# z1[0] = 0.5*1 - 0.2*2 + 0.1*(-1) + 0 = 0.5 - 0.4 - 0.1 = 0.0
# z1[1] = 0.3*1 + 0.8*2 - 0.5*(-1) + 0.5 = 0.3 + 1.6 + 0.5 + 0.5 = 2.9
print(f"\nVerificación manual:")
print(f"  z1[0] = 0.5·1 - 0.2·2 + 0.1·(-1) + 0 = {0.5*1 - 0.2*2 + 0.1*(-1) + 0}")
print(f"  z1[1] = 0.3·1 + 0.8·2 - 0.5·(-1) + 0.5 = {0.3*1 + 0.8*2 - 0.5*(-1) + 0.5}")
```
</details>

---
### Ejercicio 4 — Cuántos parámetros tiene mi red

**Enunciado:** Implementa `contar_parametros(capas)` que devuelve el número total de parámetros (pesos + biases) de un MLP totalmente conectado.

Calcula y compara:
- Red A: [784, 128, 10]
- Red B: [784, 512, 256, 128, 10]
- Red C: [784, 1024, 1024, 1024, 10]
- AlexNet aproximado: [150528, 4096, 4096, 1000]

¿Cuánta RAM ocupa cada una en float32 (4 bytes por parámetro)?

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 4</summary>

```python
def contar_parametros(capas):
    total = 0
    detalle = []
    for i in range(len(capas) - 1):
        pesos = capas[i] * capas[i+1]
        biases = capas[i+1]
        total += pesos + biases
        detalle.append((f'Capa {i+1} ({capas[i]}→{capas[i+1]})', pesos, biases))
    return total, detalle

redes = [
    ('Red A (pequeña)',   [784, 128, 10]),
    ('Red B (mediana)',   [784, 512, 256, 128, 10]),
    ('Red C (grande)',    [784, 1024, 1024, 1024, 10]),
    ('AlexNet FC (aprox)',[150528, 4096, 4096, 1000]),
]

print(f"{'Red':25} {'Parámetros':>15} {'RAM (MB)':>10}")
print('-' * 55)
for nombre, capas in redes:
    total, _ = contar_parametros(capas)
    mb = total * 4 / 1024 / 1024
    print(f"{nombre:25} {total:>15,} {mb:>10.2f}")

print("\nDetalle de Red B:")
_, detalle = contar_parametros([784, 512, 256, 128, 10])
for capa, pesos, biases in detalle:
    print(f"  {capa}: {pesos:,} pesos + {biases} biases = {pesos+biases:,}")
```
</details>

---
### Ejercicio 5 (reto) — MLP completo con forward pass por lotes

**Enunciado:** Extiende la clase `RedNeuronal` del cuerpo de la clase para que:

1. Acepte un batch (matriz de muestras) en lugar de un solo vector
2. Calcule el forward pass para todas las muestras a la vez (operaciones matriciales)
3. Implemente `cross_entropy_loss(y_pred, y_true)` para multiclase
4. Prueba con un batch de 32 muestras de 4 features, 3 clases
5. Verifica que la suma de probabilidades de cada muestra = 1

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 5</summary>

```python
import numpy as np

class MLPBatch:
    def __init__(self, capas):
        self.pesos, self.biases = [], []
        for i in range(len(capas)-1):
            W = np.random.randn(capas[i], capas[i+1]) * np.sqrt(2.0/capas[i])
            b = np.zeros(capas[i+1])
            self.pesos.append(W)
            self.biases.append(b)

    def forward(self, X):
        # X: (batch_size, n_features)
        a = X
        for i, (W, b) in enumerate(zip(self.pesos, self.biases)):
            z = a @ W + b                 # (batch, out)
            es_ultima = (i == len(self.pesos)-1)
            if es_ultima:
                z -= z.max(axis=1, keepdims=True)  # estabilidad numérica
                e = np.exp(z)
                a = e / e.sum(axis=1, keepdims=True)  # softmax por fila
            else:
                a = np.maximum(0, z)  # ReLU
        return a  # (batch, n_clases)

    @staticmethod
    def cross_entropy_loss(y_pred, y_true):
        # y_true: one-hot (batch, n_clases)
        # y_pred: probabilidades (batch, n_clases)
        eps = 1e-9
        return -np.mean(np.sum(y_true * np.log(y_pred + eps), axis=1))


np.random.seed(42)
batch_size = 32
n_features = 4
n_clases   = 3

red_batch = MLPBatch([n_features, 8, 8, n_clases])
X_batch   = np.random.randn(batch_size, n_features)
y_int     = np.random.randint(0, n_clases, batch_size)
y_onehot  = np.eye(n_clases)[y_int]  # one-hot

y_pred = red_batch.forward(X_batch)

print(f"X_batch shape: {X_batch.shape}")
print(f"y_pred shape:  {y_pred.shape}")
print(f"\nPrimeras 3 probabilidades:")
print(y_pred[:3].round(4))
print(f"\nSuma por muestra (deben ser ≈1): {y_pred.sum(axis=1)[:5].round(6)}")

loss = MLPBatch.cross_entropy_loss(y_pred, y_onehot)
print(f"\nCross-entropy loss: {loss:.4f}")
print(f"Loss de un modelo aleatorio (3 clases): log(3)={np.log(3):.4f}")
```
</details>

---
## Resumen

```python
import numpy as np

# Una neurona: combinación lineal + activación
z = np.dot(W, x) + b          # suma ponderada
a = np.maximum(0, z)          # ReLU

# Funciones de activación
relu    = lambda z: np.maximum(0, z)         # capas ocultas → defecto
sigmoid = lambda z: 1/(1+np.exp(-z))         # salida binaria (0,1)
softmax = lambda z: np.exp(z)/np.exp(z).sum()  # salida multiclase
tanh    = lambda z: np.tanh(z)               # RNN

# Forward pass (batch)
# X: (batch, features), W: (features, neuronas), b: (neuronas,)
z = X @ W + b                 # (batch, neuronas)
a = np.maximum(0, z)          # ReLU sobre todo el batch

# Regla de oro de activaciones:
# Capas ocultas → ReLU (o LeakyReLU, GELU en transformers)
# Salida binaria → Sigmoid
# Salida multiclase → Softmax
# Salida regresión → ninguna (lineal)
```

## ¿Qué viene en la próxima clase?
En la **Clase 3** aprenderemos **Backpropagation**: el algoritmo que ajusta todos los pesos de la red usando el gradiente de la pérdida. Es el motor del aprendizaje en Deep Learning.